In [ ]:
!pip install torch transformers accelerate datasets scikit-learn numpy scipy
!caffeinate -t 3600
import argparse
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset, concatenate_datasets
import matplotlib.pyplot as plt
import os
import random

# For Kolmogorov-Smirnov
from scipy.stats import ks_2samp


#################################
# Parse Arguments
#################################
def parse_args():
    parser = argparse.ArgumentParser(
        description="Run a more realistic drift experiment with domain shift, random subset of weights, and K-S drift detection."
    )
    parser.add_argument("--model_name", type=str,
                        default="gpt2",
                        help="HuggingFace model name")
    parser.add_argument("--wiki_dataset_name", type=str, default="wikitext",
                        help="Name of Wikipedia-based dataset")
    parser.add_argument("--wiki_dataset_config", type=str,
                        default="wikitext-2-raw-v1", help="Wikipedia dataset config")
    parser.add_argument("--news_dataset_name", type=str, default="cnn_dailymail",
                        help="Name of news-based dataset for domain drift")
    parser.add_argument("--news_dataset_config", type=str,
                        default="3.0.0", help="News dataset config")
    parser.add_argument("--split", type=str, default="train",
                        help="Dataset split")
    parser.add_argument("--max_texts_wiki", type=int, default=20000,
                        help="Max number of texts from Wikipedia dataset")
    parser.add_argument("--max_texts_news", type=int, default=20000,
                        help="Max number of texts from News dataset")
    parser.add_argument("--batch_size", type=int, default=64,
                        help="Batch size")
    parser.add_argument("--drift_start_batch", type=int, default=50,
                        help="Batch index at which drift starts (in gradual transition)")
    parser.add_argument("--drift_end_batch", type=int, default=100,
                        help="Batch index at which drift ends (full switch to new domain)")
    parser.add_argument("--embedding_dim", type=int, default=2000,
                        help="Dimensionality for random projection subset of params")
    parser.add_argument("--ks_alpha", type=float, default=0.05,
                        help="Significance level for K-S test")
    parser.add_argument("--window_size", type=int, default=20,
                        help="Rolling window size for drift detection")
    parser.add_argument("--output_dir", type=str, default="results_realistic",
                        help="Directory to save results and plots")

    args, _ = parser.parse_known_args()
    return args

args = parse_args()

#################################
# Setup Device
#################################
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("Using device:", device)

os.makedirs(args.output_dir, exist_ok=True)

#################################
# Load Model & Tokenizer
#################################
print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(args.model_name)
model = AutoModelForCausalLM.from_pretrained(args.model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.to(device)
model.eval()
print("Model loaded on:", device)

#################################
# Load WikiText & News Datasets
#################################
print("Loading Wikipedia dataset...")
wiki_ds = load_dataset(
    args.wiki_dataset_name,
    args.wiki_dataset_config,
    split=args.split
)
wiki_texts = wiki_ds["text"]
if args.max_texts_wiki > 0 and args.max_texts_wiki < len(wiki_texts):
    wiki_texts = wiki_texts[:args.max_texts_wiki]

print("Loading CNN/DailyMail dataset...")
news_ds = load_dataset(
    args.news_dataset_name,
    args.news_dataset_config,
    split=args.split
)
news_texts = news_ds["article"]  # 'article' field for cnn_dailymail
if args.max_texts_news > 0 and args.max_texts_news < len(news_texts):
    news_texts = news_texts[:args.max_texts_news]

print(f"Wikipedia texts: {len(wiki_texts)}")
print(f"News texts: {len(news_texts)}")

#################################
# Combine & Simulate Gradual Drift
#################################
# We'll do something simpler: 
# - The first part of the dataset is purely WikiText, 
# - Then we *gradually* mix more and more news data, 
# - Finally we end with purely news.

# Let's define a combined dataset for demonstration.
# We'll define enough total batches so that drift occurs around 
# drift_start_batch to drift_end_batch.

combined_texts = []
num_batches = (len(wiki_texts) // args.batch_size) + (len(news_texts) // args.batch_size)
print(f"Total approximate number of batches: {num_batches}")

# We'll just create a combined list now. 
# For real streaming, you'd merge them in real time, but let's do a simple static shuffle. 
# 1) Start with purely wiki texts for initial batches
# 2) Gradual shift from wiki to news 
# 3) End with purely news
# For demonstration, let's do something approximate.

# We'll slice the wiki_texts in half for the "pre-drift" portion
# then create a transition chunk mixing wiki & news,
# then end with a chunk of news.
split1 = len(wiki_texts) // 2  # mid of wiki
wiki_pre = wiki_texts[:split1]
wiki_remaining = wiki_texts[split1:]

# Let's slice some portion of news for the mixing period
split2 = len(news_texts) // 2
news_pre = news_texts[:split2]
news_remaining = news_texts[split2:]

# Combine them in an order: 
#  - full wiki_pre (pre-drift)
#  - then an interleaved portion of wiki_remaining & news_pre
#  - then final portion (news_remaining)
mixed_portion = []
min_len = min(len(wiki_remaining), len(news_pre))
for i in range(min_len):
    # Interleave
    if i % 2 == 0:
        mixed_portion.append(wiki_remaining[i])
    else:
        mixed_portion.append(news_pre[i])

combined_texts = wiki_pre + mixed_portion + news_remaining

print(f"Combined texts: {len(combined_texts)}")

# Adjust if we exceed some huge number
MAX_COMBINED = args.max_texts_wiki + args.max_texts_news
if len(combined_texts) > MAX_COMBINED:
    combined_texts = combined_texts[:MAX_COMBINED]

#################################
# Batching
#################################
def batch_generator(data, batch_size=32):
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

def encode_batch(batch_texts):
    encoding = tokenizer(
        batch_texts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=64
    )
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    return input_ids, attention_mask


#################################
# Random Subset of Model Weights
#################################
# We'll randomly pick a set of parameter indices to track, 
# and then reduce them further with a random projection or just store them raw.

# Collect all parameters into a single flat vector
params_init = []
for p in model.parameters():
    # We only want .data, not gradient
    params_init.append(p.data.cpu().numpy().ravel())
params_init = np.concatenate(params_init)
total_param_count = len(params_init)
print(f"Total parameter count = {total_param_count}")

# Choose random indices to track
subset_size = min(args.embedding_dim, total_param_count)
random_indices = np.random.choice(total_param_count, size=subset_size, replace=False)
random_indices = np.sort(random_indices)

def get_tracked_params(model):
    """
    Return the flattened parameter vector *only for the chosen subset of indices*.
    """
    flat_params = []
    for p in model.parameters():
        flat_params.append(p.data.cpu().numpy().ravel())
    flat_params = np.concatenate(flat_params)
    return flat_params[random_indices]


#################################
# Model (Optional) Fine-Tuning Setup
#################################
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=1e-5)
model.train()

# For demonstration, let's do a small step of fine-tuning 
# on every batch. In a real setting, you might do less frequent updates 
# or a more advanced training routine.

#################################
# Rolling Window K-S Test
#################################
# We'll keep a rolling buffer of the last N parameter vectors 
# and compare the distribution of the new vector to the distribution 
# of some reference (e.g., older buffer).
window = []
window_size = args.window_size

drift_flags = []
batch_num = 0

all_ks_pvals = []
all_ks_stats = []

# We define a function for K-S test (2-sample):
def ks_drift_test(ref_samples, new_samples, alpha=0.05):
    """
    Compare distribution of ref_samples vs new_samples 
    with the two-sample Kolmogorov-Smirnov test.
    Returns (statistic, p_value, is_drift).
    """
    stat, p_val = ks_2samp(ref_samples, new_samples)
    is_drift = (p_val < alpha)
    return stat, p_val, is_drift

#################################
# Main Loop
#################################
drift_detected_batches = []

for batch_texts in batch_generator(combined_texts, batch_size=args.batch_size):
    if len(batch_texts) == 0:
        break
    
    # 1) Snapshot model params before update (for demonstration)
    param_vec_before = get_tracked_params(model)

    # 2) Train (tiny step) on the current batch
    input_ids, attention_mask = encode_batch(batch_texts)
    outputs = model(input_ids, attention_mask=attention_mask, labels=input_ids)
    loss_value = outputs.loss
    optimizer.zero_grad()
    loss_value.backward()
    optimizer.step()

    # 3) Snapshot after update
    param_vec_after = get_tracked_params(model)

    # 4) Add to rolling window
    window.append(param_vec_after)
    if len(window) > window_size:
        window.pop(0)  # remove oldest

    # 5) Perform K-S test with reference distribution
    #    We compare the new distribution to either:
    #    - the entire rolling window so far (excluding the last),
    #    - or a portion of the earliest "stable" period if you want a stable baseline.
    if len(window) >= window_size:
        ref_samples = window[:-1]  # all but last
        new_samples = window[-1]   # the latest param vector

        # Flatten references into a single 1D array for simplicity.
        # Alternatively, you could do multiple tests across subsets, or average them.
        ref_samples_flat = np.concatenate(ref_samples)
        ks_stat, ks_pval, is_drift = ks_drift_test(ref_samples_flat, new_samples, alpha=args.ks_alpha)

        all_ks_stats.append(ks_stat)
        all_ks_pvals.append(ks_pval)

        if is_drift:
            drift_detected_batches.append(batch_num)
            print(f"[DRIFT DETECTED] Batch {batch_num}: KS statistic={ks_stat:.4f}, p-value={ks_pval:.6f}")
        else:
            print(f"Batch {batch_num}: KS statistic={ks_stat:.4f}, p-value={ks_pval:.6f}, no drift.")
    else:
        # Not enough samples for test yet
        all_ks_stats.append(0)
        all_ks_pvals.append(1.0)

    batch_num += 1

#################################
# Save Results & Plot
#################################
drift_output_path = os.path.join(args.output_dir, "drift_batches.npy")
np.save(drift_output_path, drift_detected_batches)

plt.figure(figsize=(12, 6))
plt.plot(all_ks_stats, label="K-S Statistic")
for d in drift_detected_batches:
    plt.axvline(x=d, color="red", linestyle="--", alpha=0.7)

plt.title("K-S Based Drift Detection on Parameter Subset")
plt.xlabel("Batch Index")
plt.ylabel("K-S Statistic")
plt.legend()
plot_path = os.path.join(args.output_dir, "ks_drift_detection.png")
plt.savefig(plot_path)
plt.show()

print(f"\n[INFO] Drift detection batches saved to: {drift_output_path}")
print(f"[INFO] Plot saved to: {plot_path}")
print("[INFO] Experiment complete.")

Using device: mps
Loading model and tokenizer...
Model loaded on: mps
Loading Wikipedia dataset...
Loading CNN/DailyMail dataset...


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Wikipedia texts: 20000
News texts: 20000
Total approximate number of batches: 624
Combined texts: 30000
Total parameter count = 124439808
Batch 19: KS statistic=0.0014, p-value=1.000000, no drift.
Batch 20: KS statistic=0.0014, p-value=1.000000, no drift.
Batch 21: KS statistic=0.0014, p-value=1.000000, no drift.
Batch 22: KS statistic=0.0014, p-value=1.000000, no drift.
Batch 23: KS statistic=0.0014, p-value=1.000000, no drift.
Batch 24: KS statistic=0.0013, p-value=1.000000, no drift.
Batch 25: KS statistic=0.0013, p-value=1.000000, no drift.
Batch 26: KS statistic=0.0013, p-value=1.000000, no drift.
Batch 27: KS statistic=0.0013, p-value=1.000000, no drift.
Batch 28: KS statistic=0.0012, p-value=1.000000, no drift.
Batch 29: KS statistic=0.0013, p-value=1.000000, no drift.
Batch 30: KS statistic=0.0012, p-value=1.000000, no drift.
Batch 31: KS statistic=0.0013, p-value=1.000000, no drift.
Batch 32: KS statistic=0.0013, p-value=1.000000, no drift.
Batch 33: KS statistic=0.0014, p-val